In [ ]:
# In[1]:

import pandas as pd
import numpy as np
import pytz
from datetime import datetime

# timezone UTC+8
tz = pytz.timezone('Asia/Shanghai')

# load files
metric_df = pd.read_csv('metric.csv')
trace_df = pd.read_csv('trace.csv')
log_df = pd.read_csv('log.csv')
error_logs_df = pd.read_csv('error_logs.csv')

# prepare a helper to convert timestamp to timezone-aware datetime in Asia/Shanghai
def attach_ts_local(df):
    # convert epoch seconds (assumed UTC) to tz-aware timestamps in Asia/Shanghai
    ts = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    df = df.copy()
    df['ts_local'] = ts
    return df

metric_df = attach_ts_local(metric_df)
trace_df = attach_ts_local(trace_df)
log_df = attach_ts_local(log_df)
error_logs_df = attach_ts_local(error_logs_df)

# define incident window in tz (inclusive start, exclusive end)
window_start = tz.localize(datetime(2021, 3, 7, 21, 30, 0))
window_end = tz.localize(datetime(2021, 3, 7, 22, 0, 0))

# Helper to aggregate per-file
def summarize_df(df, kpi_col_name):
    # rename KPI column to unified name
    df2 = df.rename(columns={kpi_col_name: 'KPI_name'})
    # ensure 'value' numeric
    df2['value'] = pd.to_numeric(df2['value'], errors='coerce')
    # global aggregates over full series (rule: compute thresholds on full series)
    grp = df2.groupby(['cmdb_id', 'KPI_name'])['value']
    agg_full = grp.agg(count_of_points='count',
                       p95_value=lambda x: float(x.quantile(0.95)))
    agg_full = agg_full.reset_index()
    # counts within window
    in_window = df2[(df2['ts_local'] >= window_start) & (df2['ts_local'] < window_end)]
    grp_win = in_window.groupby(['cmdb_id', 'KPI_name'])['value'].count().reset_index().rename(columns={'value':'count_in_window'})
    # merge, fill missing window counts with 0
    merged = agg_full.merge(grp_win, on=['cmdb_id','KPI_name'], how='left')
    merged['count_in_window'] = merged['count_in_window'].fillna(0).astype(int)
    # sort and keep top 20 later
    merged = merged.sort_values(by=['count_in_window','count_of_points'], ascending=[False, False])
    return merged

metric_summary = summarize_df(metric_df, 'kpi_name')
trace_summary = summarize_df(trace_df, 'trace_name')
log_summary = summarize_df(log_df, 'log_name')

# distinct cmdb_id lists (as sorted Series for compact display)
metric_cmdbs = pd.Series(sorted(metric_df['cmdb_id'].dropna().unique()), name='metric_cmdb_id')
trace_cmdbs  = pd.Series(sorted(trace_df['cmdb_id'].dropna().unique()), name='trace_cmdb_id')
log_cmdbs    = pd.Series(sorted(log_df['cmdb_id'].dropna().unique()), name='log_cmdb_id')

# compact outputs: top 20 rows each and the cmdb lists
metric_summary_head = metric_summary[['cmdb_id','KPI_name','count_of_points','p95_value','count_in_window']].head(20)
trace_summary_head  = trace_summary[['cmdb_id','KPI_name','count_of_points','p95_value','count_in_window']].head(20)
log_summary_head    = log_summary[['cmdb_id','KPI_name','count_of_points','p95_value','count_in_window']].head(20)

metric_summary_head, trace_summary_head, log_summary_head, metric_cmdbs, trace_cmdbs, log_cmdbs

```
Out[1]:
```
```python
# Summarize results in plain English based on previous execution outputs
summary = (
    "Summary of telemetry aggregation (window = 2021-03-07 21:30:00 to 22:00:00 Asia/Shanghai):\n\n"
    "1) Metrics (metric.csv):\n"
    "- Top groups are dominated by IG01 and IG02 KPIs. Many metric groups have full-series counts = 31 and "
    "count_in_window = 30, i.e., almost all points fall inside the incident window.\n"
    "- Example p95 values (from top rows): HeapMemoryMax ~2.04e9, HeapMemoryUsed ~8.1e8, CPU load ~1.2.\n"
    "- Distinct cmdb_id count for metric.csv: 29 (includes IG01, IG02, MG01/02, Mysql01, apache01/02, dockerA1/2, dockerB1/2, ...).\n\n"
    "2) Traces (trace.csv):\n"
    "- Trace groups for IG01 and IG02 show count_of_points = 30 and count_in_window = 30 (all trace points fall in the window).\n"
    "- Notable p95s: trace.self.duration_p95 ~60s; trace.self.duration_mean ~6.4-6.9s; large row_count p95s (~885-937).\n"
    "- Distinct cmdb_id list for trace.csv: 12 (includes IG01, IG02, Tomcat01-04, dockerA/B...).\n\n"
    "3) Logs (log.csv):\n"
    "- Many Tomcat and apache log metrics have count_of_points = 31 and count_in_window = 30 (they cover the window).\n"
    "- log.error_count p95 is 0.0 for most Tomcats; apache error_count p95 is non-zero (examples: apache01 ~11.5, apache02 ~13.0).\n"
    "- Some log metrics (gc_pause_max) have fewer total points (e.g., 9, 8, 23) and correspondingly smaller count_in_window values.\n"
    "- Distinct cmdb_id list for log.csv: 6 (Mysql01, Tomcat01-04, apache01, apache02).\n\n"
    "Overall conclusion: IG01 and IG02 telemetry (metrics and traces) fully cover the incident window and show high p95 values for some JVM/trace KPIs; logs show low error_count for Tomcats but elevated error_count p95 for apache instances. "
    "These are the compact, top-level findings from the aggregated summaries."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(    cmdb_id                                           KPI_name  count_of_points     p95_value  count_in_window
0      IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax               31  2.040136e+09               30
2      IG01          JVM-Memory_7778_JVM_Memory_HeapMemoryUsed               31  8.097002e+08               30
3      IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed               31  5.815186e+07               30
6      IG01           JVM-Threads_7778_JVM_ThreadCount_Threads               31  7.200000e+01               30
12     IG01                            OSLinux-CPU_CPU_CPULoad               31  1.215000e+00               30
13     IG01                         OSLinux-CPU_CPU_CPUSysTime               31  4.553100e-01               30
14     IG01                        OSLinux-CPU_CPU_CPUUserTime               31  2.526614e+01               30
15     IG01                             OSLinux-CPU_CPU_CPUWio               31  2.786080e-01               30
16     IG01                        OSLinux-CPU_CPU_CPUidleutil               31  7.445946e+01               30
66     IG01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT               31  0.000000e+00               30
67     IG01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT               31  4.850000e+01               30
77     IG01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies               31  1.000000e+00               30
80     IG01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...               31  3.000000e-01               30
81     IG01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount               31  3.000000e+00               30
88     IG01                 OSLinux-OSLinux_ZABBIX_Host_Uptime               31  6.324527e+06               30
89     IG02           JVM-Memory_7778_JVM_Memory_HeapMemoryMax               31  2.040136e+09               30
91     IG02          JVM-Memory_7778_JVM_Memory_HeapMemoryUsed               31  8.375575e+08               30
92     IG02        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed               31  5.806961e+07               30
95     IG02           JVM-Threads_7778_JVM_ThreadCount_Threads               31  7.350000e+01               30
101    IG02                            OSLinux-CPU_CPU_CPULoad               31  1.205000e+00               30,    cmdb_id                         KPI_name  count_of_points   p95_value  count_in_window
0     IG01         trace.self.duration_mean               30    6.461165               30
1     IG01          trace.self.duration_p95               30   60.082950               30
2     IG01             trace.self.row_count               30  885.500000               30
3     IG01  trace.to_Tomcat01.duration_mean               30    0.358536               30
4     IG01   trace.to_Tomcat01.duration_p95               30    1.879535               30
5     IG01      trace.to_Tomcat01.row_count               30  121.400000               30
6     IG01  trace.to_Tomcat02.duration_mean               30    0.374186               30
7     IG01   trace.to_Tomcat02.duration_p95               30    1.355960               30
8     IG01      trace.to_Tomcat02.row_count               30  109.200000               30
9     IG01  trace.to_Tomcat03.duration_mean               30    0.280226               30
10    IG01   trace.to_Tomcat03.duration_p95               30    1.292020               30
11    IG01      trace.to_Tomcat03.row_count               30  118.500000               30
12    IG01  trace.to_Tomcat04.duration_mean               30    0.348811               30
13    IG01   trace.to_Tomcat04.duration_p95               30    1.251712               30
14    IG01      trace.to_Tomcat04.row_count               30  117.200000               30
15    IG02         trace.self.duration_mean               30    6.868338               30
16    IG02          trace.self.duration_p95               30   60.769210               30
17    IG02             trace.self.row_count               30  937.300000               30
18    IG02  trace.to_Tomcat01.duration_mean               30    0.358555               30
19    IG02   trace.to_Tomcat01.duration_p95               30    2.051408               30,      cmdb_id          KPI_name  count_of_points  p95_value  count_in_window
2   Tomcat01   log.error_count               31       0.00               30
4   Tomcat01     log.row_count               31     222.00               30
5   Tomcat02   log.error_count               31       0.00               30
7   Tomcat02     log.row_count               31     238.00               30
11  Tomcat04   log.error_count               31       0.00               30
13  Tomcat04     log.row_count               31     227.50               30
14  apache01   log.error_count               31      11.50               30
15  apache01     log.row_count               31     450.00               30
16  apache02   log.error_count               31      13.00               30
17  apache02     log.row_count               31     444.50               30
8   Tomcat03   log.error_count               30       0.00               30
10  Tomcat03     log.row_count               30     229.05               30
6   Tomcat02  log.gc_pause_max               23       0.12               23
0    Mysql01   log.error_count               13       0.00               13
1    Mysql01     log.row_count               13       4.00               13
3   Tomcat01  log.gc_pause_max                9       0.08                9
9   Tomcat03  log.gc_pause_max                9       0.09                9
12  Tomcat04  log.gc_pause_max                8       0.08                8, 0         IG01
1         IG02
2         MG01
3         MG02
4      Mysql01
        ...   
24    apache02
25    dockerA1
26    dockerA2
27    dockerB1
28    dockerB2
Name: metric_cmdb_id, Length: 29, dtype: object, 0         IG01
1         IG02
2         MG01
3         MG02
4     Tomcat01
5     Tomcat02
6     Tomcat03
7     Tomcat04
8     dockerA1
9     dockerA2
10    dockerB1
11    dockerB2
Name: trace_cmdb_id, dtype: object, 0     Mysql01
1    Tomcat01
2    Tomcat02
3    Tomcat03
4    Tomcat04
5    apache01
6    apache02
Name: log_cmdb_id, dtype: object)```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np
import pytz
from datetime import datetime

# Reuse existing metric_df if present; otherwise load
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')
    # attach tz-aware local timestamp column
    metric_df['ts_local'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(pytz.timezone('Asia/Shanghai'))

# ensure necessary columns and types
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')
if 'ts_local' not in metric_df.columns:
    metric_df['ts_local'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(pytz.timezone('Asia/Shanghai'))

tz = pytz.timezone('Asia/Shanghai')
window_start = tz.localize(datetime(2021, 3, 7, 21, 30, 0))
window_end   = tz.localize(datetime(2021, 3, 7, 22, 0, 0))

# helper to compute anomalies given a quantile level
def find_anomalies(metric_df, quantile_level=0.95):
    # compute global threshold (quantile) per (cmdb_id, kpi_name) over full series
    thresholds = (metric_df
                  .groupby(['cmdb_id','kpi_name'])['value']
                  .quantile(quantile_level)
                  .reset_index()
                  .rename(columns={'value': 'threshold'}))
    # filter window
    metric_win = metric_df[(metric_df['ts_local'] >= window_start) & (metric_df['ts_local'] < window_end)].copy()
    # merge thresholds into window
    metric_win = metric_win.merge(thresholds, on=['cmdb_id','kpi_name'], how='left')
    # mark anomalies where value >= threshold (threshold may be NaN if group missing globally)
    metric_win['is_anom'] = metric_win['value'] >= metric_win['threshold']
    # aggregate per (cmdb_id, kpi_name)
    def earliest_anom_ts(s_ts, s_is_anom):
        if s_is_anom.any():
            return s_ts[s_is_anom].min()
        return pd.NaT
    agg = (metric_win
           .groupby(['cmdb_id','kpi_name'])
           .apply(lambda g: pd.Series({
               'count_anomalies': int(g['is_anom'].sum()),
               'earliest_anomaly_ts': earliest_anom_ts(g['ts_local'], g['is_anom']),
               'max_value_in_window': float(g['value'].max()) if len(g)>0 else np.nan,
               'p_threshold': float(g['threshold'].iloc[0]) if len(g)>0 else np.nan
           }))
           .reset_index())
    # keep only groups with anomalies
    anomalies = agg[agg['count_anomalies'] > 0].copy()
    if anomalies.empty:
        return anomalies, metric_win  # no anomalies
    # compute severity_pct, handle zero threshold
    def calc_sev(maxv, thr):
        if pd.isna(thr) or thr == 0:
            return np.nan
        return 100.0 * (maxv - thr) / thr
    anomalies['severity_pct'] = anomalies.apply(lambda r: calc_sev(r['max_value_in_window'], r['p_threshold']), axis=1)
    # format earliest_anomaly_ts as string in Asia/Shanghai
    def fmt_ts(ts):
        if pd.isna(ts) or ts is pd.NaT:
            return pd.NaT
        # ensure tz-aware Asia/Shanghai
        if ts.tzinfo is None:
            ts = ts.tz_localize(pytz.timezone('Asia/Shanghai'))
        else:
            ts = ts.tz_convert(pytz.timezone('Asia/Shanghai'))
        return ts.strftime('%Y-%m-%d %H:%M:%S%z')
    anomalies['earliest_anomaly_timestamp'] = anomalies['earliest_anomaly_ts'].apply(fmt_ts)
    # select and order columns
    anomalies = anomalies[['cmdb_id','kpi_name','count_anomalies','earliest_anomaly_timestamp',
                           'max_value_in_window','p_threshold','severity_pct']].rename(columns={'p_threshold':'p_value'})
    # sort by severity desc then count desc
    anomalies = anomalies.sort_values(by=['severity_pct','count_anomalies'], ascending=[False, False])
    return anomalies, metric_win

# Try P95 first
anomalies_p95, metric_win = find_anomalies(metric_df, 0.95)
relaxed_to = None
if anomalies_p95.empty:
    # relax to P90
    anomalies_p90, metric_win = find_anomalies(metric_df, 0.90)
    anomalies = anomalies_p90
    relaxed_to = 'P90'
else:
    anomalies = anomalies_p95
    relaxed_to = 'P95'

# Limit to top 20 rows and ensure compact columns
anomalies_head = anomalies.head(20).reset_index(drop=True)

# Per-component summary
if not anomalies.empty:
    comp_summary = (anomalies
                    .groupby('cmdb_id')
                    .agg(total_anomaly_points_in_window=('count_anomalies','sum'),
                         number_of_KPIs_with_anomalies=('kpi_name','nunique'))
                    .reset_index()
                    .query("total_anomaly_points_in_window > 0")
                    .sort_values(by='total_anomaly_points_in_window', ascending=False)
                    .head(20))
else:
    comp_summary = anomalies.groupby('cmdb_id').size().reset_index(name='n') # empty placeholder
    comp_summary = comp_summary.iloc[0:0]  # empty DF

# Provide outputs: anomalies_head, comp_summary, and which threshold was used
anomalies_head, comp_summary, relaxed_to

```
Out[2]:
```
# Create a plain-English summary string based on the previously computed results
try:
    top_anoms = anomalies_head.head(5)
    top_comps = comp_summary.head(6)
    thr_used = relaxed_to
except NameError:
    top_anoms = None
    top_comps = None
    thr_used = None

if top_anoms is None or top_comps is None or thr_used is None:
    summary = "No previous results found in the kernel to summarize."
else:
    lines = []
    lines.append(f"Anomaly summary for metric.csv within 2021-03-07 21:30:00–22:00:00 (Asia/Shanghai).")
    lines.append(f"Global threshold used: {thr_used} (computed from full series before window filtering).")
    lines.append("")
    lines.append("Top anomaly candidates (by severity):")
    for i, row in top_anoms.head(5).iterrows():
        sev = row['severity_pct']
        sev_str = f"{sev:.1f}%" if pd.notna(sev) else "N/A"
        lines.append(
            f"- {row['cmdb_id']} | {row['kpi_name']} : "
            f"count_anomalies={int(row['count_anomalies'])}, "
            f"earliest={row['earliest_anomaly_timestamp']}, "
            f"max_in_window={row['max_value_in_window']}, "
            f"p{thr_used.lower()}={row['p_value']}, severity={sev_str}"
        )
    lines.append("")
    lines.append("Per-component anomaly summary (top components by total anomaly points):")
    for i, row in top_comps.iterrows():
        lines.append(
            f"- {row['cmdb_id']}: total_anomaly_points_in_window={int(row['total_anomaly_points_in_window'])}, "
            f"number_of_KPIs_with_anomalies={int(row['number_of_KPIs_with_anomalies'])}"
        )
    lines.append("")
    lines.append("High-level takeaway:")
    lines.append(
        "Multiple components exhibit many KPI-level anomalies in the incident window. "
        "Database hosts (Mysql01, Mysql02) top the list with the largest number of anomaly points across many KPIs, "
        "followed by Redis and several Tomcat/apache nodes. The top KPI-level anomalies include CPU I/O wait spikes, "
        "large InnoDB row lock times, and elevated TCP CLOSE-WAIT counts—suggesting I/O/DB contention and networking/socket pressure during the window."
    )
    summary = "\n".join(lines)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(         cmdb_id                                           kpi_name  count_anomalies earliest_anomaly_timestamp  max_value_in_window     p_value  severity_pct
0        Redis02                             OSLinux-CPU_CPU_CPUWio                2   2021-03-07 21:33:00+0800               0.3875    0.017503   2113.906188
1        Mysql01              Mysql-MySQL_3306_Innodb Row Lock Time                2   2021-03-07 21:46:00+0800              16.3167    2.722530    499.321220
2       apache02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT                2   2021-03-07 21:37:00+0800               2.0000    0.550000    263.636364
3       Tomcat02                             OSLinux-CPU_CPU_CPUWio                1   2021-03-07 21:31:00+0800               0.4491    0.133035    237.580336
4        Mysql02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKPer...                1   2021-03-07 21:55:00+0800               0.8000    0.240000    233.333333
5       apache02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps                1   2021-03-07 21:40:00+0800               4.0000    1.200000    233.333333
6           IG01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps                1   2021-03-07 21:32:00+0800               5.0000    1.500000    233.333333
7       Tomcat03                             OSLinux-CPU_CPU_CPUWio                1   2021-03-07 21:33:00+0800               1.0398    0.328530    216.500776
8        Mysql02              Mysql-MySQL_3306_Innodb Row Lock Time                1   2021-03-07 21:51:00+0800               1.7667    0.563335    213.614457
9   ServiceTest5                                            app.cnt                2   2021-03-07 21:42:00+0800             306.0000  115.000000    166.086957
10       Mysql01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps                1   2021-03-07 21:54:00+0800               9.0000    3.400000    164.705882
11       Mysql01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps                1   2021-03-07 21:39:00+0800               7.0000    2.800000    150.000000
12      Tomcat01  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...                4   2021-03-07 21:43:00+0800               5.0000    2.000000    150.000000
13      apache01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps                1   2021-03-07 21:45:00+0800              28.6833   11.509990    149.203518
14      Tomcat02  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...                2   2021-03-07 21:34:00+0800               6.0000    2.500000    140.000000
15       Redis01                             OSLinux-CPU_CPU_CPUWio                3   2021-03-07 21:30:00+0800               0.0499    0.020800    139.903846
16       Mysql02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead                1   2021-03-07 21:40:00+0800             133.4667   55.906700    138.731136
17      Tomcat01                             OSLinux-CPU_CPU_CPUWio                1   2021-03-07 21:46:00+0800               0.7776    0.339080    129.326413
18      Tomcat01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps                1   2021-03-07 21:37:00+0800               1.0000    0.450000    122.222222
19      apache01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKTps                1   2021-03-07 21:41:00+0800               1.0000    0.450000    122.222222,           cmdb_id  total_anomaly_points_in_window  number_of_KPIs_with_anomalies
4         Mysql01                            2533                            221
5         Mysql02                            2114                            215
7         Redis02                            1364                            107
6         Redis01                            1335                            109
22       Tomcat04                            1025                            110
20       Tomcat02                             967                            107
21       Tomcat03                             962                            108
19       Tomcat01                             866                            105
2            MG01                             676                             89
3            MG02                             667                             91
0            IG01                             624                             89
24       apache02                             615                             81
1            IG02                             570                             83
23       apache01                             150                             37
12   ServiceTest3                              54                              4
10  ServiceTest11                              54                              4
9   ServiceTest10                              54                              4
8    ServiceTest1                              54                              4
11   ServiceTest2                              54                              4
13   ServiceTest4                              54                              4, 'P95')```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import pytz
from datetime import datetime

# reuse trace_df if available; otherwise load
try:
    trace_df
except NameError:
    trace_df = pd.read_csv('trace.csv')

# ensure types and tz-aware local timestamp in Asia/Shanghai
tz = pytz.timezone('Asia/Shanghai')
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')
if 'ts_local' not in trace_df.columns:
    trace_df['ts_local'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    # ensure tz-aware and converted
    trace_df['ts_local'] = pd.to_datetime(trace_df['ts_local']).dt.tz_localize(None).dt.tz_localize(tz)

# define incident window (inclusive start, exclusive end)
window_start = tz.localize(datetime(2021, 3, 7, 21, 30, 0))
window_end = tz.localize(datetime(2021, 3, 7, 22, 0, 0))

def analyze_traces(df, quantile_level=0.95):
    # compute global quantile per (cmdb_id, trace_name) on full series
    thresholds = (df
                  .groupby(['cmdb_id','trace_name'])['value']
                  .quantile(quantile_level)
                  .reset_index()
                  .rename(columns={'value': 'p_value'}))
    # filter to window
    df_win = df[(df['ts_local'] >= window_start) & (df['ts_local'] < window_end)].copy()
    # merge thresholds into window
    df_win = df_win.merge(thresholds, on=['cmdb_id','trace_name'], how='left')
    # mark anomalies where value >= p_value (NaN p_value -> False)
    df_win['is_anom'] = df_win['value'] >= df_win['p_value']
    # aggregate per (cmdb_id, trace_name)
    def earliest_anom_ts(series_ts, series_is_anom):
        if series_is_anom.any():
            return series_ts[series_is_anom].min()
        return pd.NaT
    agg = (df_win
           .groupby(['cmdb_id','trace_name'])
           .apply(lambda g: pd.Series({
               'count_anomalies': int(g['is_anom'].sum()),
               'earliest_anomaly_ts': earliest_anom_ts(g['ts_local'], g['is_anom']),
               'max_value_in_window': float(g['value'].max()) if len(g)>0 else np.nan,
               'p_value': float(g['p_value'].iloc[0]) if len(g)>0 else np.nan
           }))
           .reset_index())
    # keep only groups with anomalies
    anomalies = agg[agg['count_anomalies'] > 0].copy()
    if anomalies.empty:
        return anomalies, df_win
    # compute severity_pct, handle zero p_value -> NaN
    def calc_sev(maxv, pval):
        if pd.isna(pval) or pval == 0:
            return np.nan
        return 100.0 * (maxv - pval) / pval
    anomalies['severity_pct'] = anomalies.apply(lambda r: calc_sev(r['max_value_in_window'], r['p_value']), axis=1)
    # format earliest_anomaly_ts as string in Asia/Shanghai
    def fmt_ts(ts):
        if pd.isna(ts) or ts is pd.NaT:
            return pd.NaT
        if ts.tzinfo is None:
            ts = tz.localize(ts)
        else:
            ts = ts.tz_convert(tz)
        return ts.strftime('%Y-%m-%d %H:%M:%S%z')
    anomalies['earliest_anomaly_timestamp'] = anomalies['earliest_anomaly_ts'].apply(fmt_ts)
    # select requested columns and sort
    anomalies = anomalies[['cmdb_id','trace_name','count_anomalies','earliest_anomaly_timestamp',
                           'max_value_in_window','p_value','severity_pct']]
    anomalies = anomalies.sort_values(by=['severity_pct','count_anomalies'], ascending=[False, False])
    return anomalies, df_win

# Try P95 first
anomalies_p95, trace_win = analyze_traces(trace_df, 0.95)
relaxed_to = 'P95'
if anomalies_p95.empty:
    anomalies_p90, trace_win = analyze_traces(trace_df, 0.90)
    anomalies = anomalies_p90
    relaxed_to = 'P90'
else:
    anomalies = anomalies_p95

# compact top-20 anomalies
anomalies_head = anomalies.head(20).reset_index(drop=True)

# per-component summary
if not anomalies.empty:
    comp_summary = (anomalies
                    .groupby('cmdb_id')
                    .agg(total_anomaly_points_in_window=('count_anomalies','sum'),
                         number_of_trace_metrics_with_anomalies=('trace_name','nunique'))
                    .reset_index()
                    .query("total_anomaly_points_in_window > 0")
                    .sort_values(by='total_anomaly_points_in_window', ascending=False)
                    .head(20))
else:
    comp_summary = anomalies.groupby('cmdb_id').size().reset_index(name='n').iloc[0:0]

# return compact results: top anomalies, per-component summary, and threshold used
anomalies_head, comp_summary, relaxed_to

```
Out[3]:
```
Summary of trace.csv analysis for the incident window 2021-03-07 21:30–22:00 (Asia/Shanghai):

Threshold used: P95 computed on the full trace series (no relaxation needed).

Top trace-level anomaly candidates (by severity):
- MG02 | trace.from_dockerB2.duration_p95 — count_anomalies=2, earliest=2021-03-07 21:30:00+0800, max=0.2342, p95=0.01952, severity ≈ 1099.8%
- dockerB2 | trace.to_MG02.duration_p95 — count_anomalies=2, earliest=2021-03-07 21:30:00+0800, max=0.2342, p95=0.01952, severity ≈ 1099.8%
- MG01 | trace.from_dockerA2.duration_p95 — count_anomalies=2, earliest=2021-03-07 21:31:00+0800, max=0.3481, p95=0.03280, severity ≈ 961.4%
- dockerA2 | trace.to_MG01.duration_p95 — count_anomalies=2, earliest=2021-03-07 21:31:00+0800, max=0.3481, p95=0.03280, severity ≈ 961.4%
- MG01 | trace.from_dockerB2.duration_p95 — count_anomalies=2, earliest=2021-03-07 21:30:00+0800, max=0.3380, p95=0.05172, severity ≈ 553.6%

Compact per-component summary (components with anomalies, sorted by total anomaly points):
- MG02: total_anomaly_points_in_window = 79, number_of_trace_metrics_with_anomalies = 39
- MG01: total_anomaly_points_in_window = 79, number_of_trace_metrics_with_anomalies = 39
- dockerB2: 55 anomaly points across 18 trace metrics
- dockerA2: 35 anomaly points across 18 trace metrics
- Tomcat01: 34 anomaly points across 18 trace metrics
- Tomcat03: 33 anomaly points across 18 trace metrics
- Tomcat04: 33 anomaly points across 18 trace metrics
- Tomcat02: 33 anomaly points across 18 trace metrics
- dockerA1: 33 anomaly points across 18 trace metrics
- dockerB1: 33 anomaly points across 18 trace metrics
- IG01: 30 anomaly points across 15 trace metrics
- IG02: 30 anomaly points across 15 trace metrics
(Top results shown; full list available in the detailed output.)

Takeaway:
- Significant trace latency spikes occurred between MG services and Docker-hosted services (MG01/MG02 and dockerA*/dockerB*), with many duration_p95/duration_mean entries exceeding their global P95 by large margins during the incident window.
- Several Tomcat nodes also show many trace anomalies. This pattern suggests elevated inter-service latency between MG and containerized services (and wider service-to-Tomcat latency) during 21:30–22:00.

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                         trace_name  count_anomalies earliest_anomaly_timestamp  max_value_in_window   p_value  severity_pct
0       MG02   trace.from_dockerB2.duration_p95                2   2021-03-07 21:30:00+0800              0.23420  0.019520   1099.795082
1   dockerB2         trace.to_MG02.duration_p95                2   2021-03-07 21:30:00+0800              0.23420  0.019520   1099.795082
2       MG01   trace.from_dockerA2.duration_p95                2   2021-03-07 21:31:00+0800              0.34810  0.032797    961.361384
3   dockerA2         trace.to_MG01.duration_p95                2   2021-03-07 21:31:00+0800              0.34810  0.032797    961.361384
4       MG01   trace.from_dockerB2.duration_p95                2   2021-03-07 21:30:00+0800              0.33800  0.051717    553.550539
5   dockerB2         trace.to_MG01.duration_p95                2   2021-03-07 21:30:00+0800              0.33800  0.051717    553.550539
6       MG01  trace.from_dockerA1.duration_mean                2   2021-03-07 21:31:00+0800              0.32945  0.074618    341.514564
7   dockerA1        trace.to_MG01.duration_mean                2   2021-03-07 21:31:00+0800              0.32945  0.074618    341.514564
8       MG02   trace.from_dockerA1.duration_p95                2   2021-03-07 21:33:00+0800              0.28930  0.082022    252.708098
9   dockerA1         trace.to_MG02.duration_p95                2   2021-03-07 21:33:00+0800              0.28930  0.082022    252.708098
10      MG01   trace.from_dockerB1.duration_p95                2   2021-03-07 21:33:00+0800              1.01340  0.299667    238.174810
11  dockerB1         trace.to_MG01.duration_p95                2   2021-03-07 21:33:00+0800              1.01340  0.299667    238.174810
12      MG02   trace.from_dockerB1.duration_p95                2   2021-03-07 21:30:00+0800              1.01300  0.306292    230.729613
13  dockerB1         trace.to_MG02.duration_p95                2   2021-03-07 21:30:00+0800              1.01300  0.306292    230.729613
14      MG01   trace.from_dockerA1.duration_p95                2   2021-03-07 21:31:00+0800              1.08750  0.353857    207.327102
15  dockerA1         trace.to_MG01.duration_p95                2   2021-03-07 21:31:00+0800              1.08750  0.353857    207.327102
16      MG02  trace.from_dockerA2.duration_mean                2   2021-03-07 21:32:00+0800              0.26100  0.103504    152.163680
17  dockerA2        trace.to_MG02.duration_mean                2   2021-03-07 21:32:00+0800              0.26100  0.103504    152.163680
18      MG01     trace.to_dockerA1.duration_p95                2   2021-03-07 21:30:00+0800              2.04250  1.040225     96.351751
19  dockerA1       trace.from_MG01.duration_p95                2   2021-03-07 21:30:00+0800              2.04250  1.040225     96.351751,      cmdb_id  total_anomaly_points_in_window  number_of_trace_metrics_with_anomalies
3       MG02                              79                                      39
2       MG01                              79                                      39
11  dockerB2                              55                                      18
9   dockerA2                              35                                      18
4   Tomcat01                              34                                      18
6   Tomcat03                              33                                      18
7   Tomcat04                              33                                      18
5   Tomcat02                              33                                      18
8   dockerA1                              33                                      18
10  dockerB1                              33                                      18
0       IG01                              30                                      15
1       IG02                              30                                      15, 'P95')```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz
from datetime import datetime

# Reuse existing metric_df if available; otherwise load and prepare
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')

# Ensure value numeric and ts_local exists and is in Asia/Shanghai
tz = pytz.timezone('Asia/Shanghai')
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')
if 'ts_local' not in metric_df.columns:
    metric_df['ts_local'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    # ensure ts_local is tz-aware in Asia/Shanghai
    metric_df['ts_local'] = pd.to_datetime(metric_df['ts_local'])
    metric_df['ts_local'] = metric_df['ts_local'].dt.tz_localize(None).dt.tz_localize(tz)

# Selected components
selected_components = [
    "Mysql01","Mysql02","Redis01","Redis02",
    "MG01","MG02","IG01","IG02",
    "Tomcat01","Tomcat02","Tomcat03","Tomcat04",
    "apache01","apache02"
]

# Incident window (inclusive start, exclusive end)
window_start = tz.localize(datetime(2021, 3, 7, 21, 30, 0))
window_end   = tz.localize(datetime(2021, 3, 7, 22, 0, 0))

# Work on selected components only (but compute thresholds on full series for those components)
metric_sel = metric_df[metric_df['cmdb_id'].isin(selected_components)].copy()

def run_detection(metric_sel, quantile_level=0.95):
    # compute global quantile per (cmdb_id,kpi_name) BEFORE filtering by time
    thresholds = (metric_sel
                  .groupby(['cmdb_id','kpi_name'])['value']
                  .quantile(quantile_level)
                  .reset_index()
                  .rename(columns={'value':'p_value'}))
    # metric points in the window
    metric_win = metric_sel[(metric_sel['ts_local'] >= window_start) & (metric_sel['ts_local'] < window_end)].copy()
    if metric_win.empty:
        return pd.DataFrame(), pd.DataFrame(), thresholds

    # merge thresholds
    metric_win = metric_win.merge(thresholds, on=['cmdb_id','kpi_name'], how='left')
    # mark anomalies: value >= p_value (if p_value NaN -> False)
    metric_win['is_anom'] = False
    mask_p_notna = metric_win['p_value'].notna()
    metric_win.loc[mask_p_notna, 'is_anom'] = metric_win.loc[mask_p_notna, 'value'] >= metric_win.loc[mask_p_notna, 'p_value']

    # aggregate per KPI within window
    def earliest_ts(s_ts, s_is):
        if s_is.any():
            return s_ts[s_is].min()
        return pd.NaT

    kpi_agg = (metric_win
               .groupby(['cmdb_id','kpi_name'])
               .apply(lambda g: pd.Series({
                   'count_anomalies': int(g['is_anom'].sum()),
                   'earliest_anomaly_ts': earliest_ts(g['ts_local'], g['is_anom']),
                   'max_value_in_window': float(g['value'].max()) if len(g)>0 else np.nan,
                   'p_value': float(g['p_value'].iloc[0]) if len(g)>0 else np.nan
               }))
               .reset_index())

    # keep only KPIs with anomalies (count_anomalies > 0)
    kpi_anoms = kpi_agg[kpi_agg['count_anomalies'] > 0].copy()
    if kpi_anoms.empty:
        return pd.DataFrame(), pd.DataFrame(), thresholds

    # compute severity_pct per KPI, handle p_value==0 by setting NaN
    def calc_sev(maxv, pval):
        if pd.isna(pval) or pval == 0:
            return np.nan
        return 100.0 * (maxv - pval) / pval

    kpi_anoms['severity_pct'] = kpi_anoms.apply(lambda r: calc_sev(r['max_value_in_window'], r['p_value']), axis=1)

    # format earliest_anomaly_ts as string in Asia/Shanghai
    def fmt_ts(ts):
        if pd.isna(ts) or ts is pd.NaT:
            return pd.NaT
        if ts.tzinfo is None:
            ts = tz.localize(ts)
        else:
            ts = ts.tz_convert(tz)
        return ts.strftime('%Y-%m-%d %H:%M:%S%z')
    kpi_anoms['earliest_anomaly_timestamp'] = kpi_anoms['earliest_anomaly_ts'].apply(fmt_ts)

    # Component summary aggregation
    comp_agg = (kpi_anoms
                .groupby('cmdb_id')
                .agg(
                    earliest_anomaly_ts_comp=('earliest_anomaly_ts','min'),
                    total_anomaly_points_in_window=('count_anomalies','sum'),
                    number_of_KPIs_with_anomalies=('kpi_name','nunique'),
                    max_severity_pct_across_KPIs=('severity_pct','max')
                )
                .reset_index())

    # format earliest ts
    comp_agg['earliest_anomaly_timestamp'] = comp_agg['earliest_anomaly_ts_comp'].apply(fmt_ts)
    comp_agg = comp_agg[['cmdb_id','earliest_anomaly_timestamp','total_anomaly_points_in_window',
                         'number_of_KPIs_with_anomalies','max_severity_pct_across_KPIs']]

    # sort by earliest_anomaly_timestamp asc, tie-breaker max_severity_pct desc
    comp_agg_sorted = comp_agg.sort_values(by=['earliest_anomaly_timestamp','max_severity_pct_across_KPIs'],
                                           ascending=[True, False]).reset_index(drop=True)

    # For Mysql01 and Mysql02: pick top KPI by severity_pct (exclude NaN severity)
    mysql_top_rows = []
    for host in ['Mysql01','Mysql02']:
        host_kpis = kpi_anoms[kpi_anoms['cmdb_id']==host].copy()
        host_kpis = host_kpis[host_kpis['severity_pct'].notna()]
        if not host_kpis.empty:
            # pick KPI with largest severity_pct, tie-breaker by count_anomalies desc then earliest timestamp
            host_kpis = host_kpis.sort_values(by=['severity_pct','count_anomalies','earliest_anomaly_ts'],
                                              ascending=[False, False, True])
            top = host_kpis.iloc[0]
            mysql_top_rows.append({
                'cmdb_id': top['cmdb_id'],
                'kpi_name': top['kpi_name'],
                'earliest_anomaly_timestamp_for_this_kpi': top['earliest_anomaly_timestamp'],
                'max_value_in_window': top['max_value_in_window'],
                'p95_value': top['p_value'],
                'severity_pct': top['severity_pct']
            })
    mysql_top_df = pd.DataFrame(mysql_top_rows, columns=['cmdb_id','kpi_name','earliest_anomaly_timestamp_for_this_kpi',
                                                         'max_value_in_window','p95_value','severity_pct'])

    return comp_agg_sorted, mysql_top_df, kpi_anoms, thresholds

# Try P95 first
comp_summary_p95, mysql_top_p95, kpi_anoms_p95, thresholds_p95 = run_detection(metric_sel, 0.95)
relaxed_to = 'P95'
if comp_summary_p95.empty:
    # Relax to P90
    comp_summary_p90, mysql_top_p90, kpi_anoms_p90, thresholds_p90 = run_detection(metric_sel, 0.90)
    comp_summary = comp_summary_p90
    mysql_top_df = mysql_top_p90
    kpi_anoms_used = kpi_anoms_p90
    thresholds_used = thresholds_p90
    relaxed_to = 'P90'
else:
    comp_summary = comp_summary_p95
    mysql_top_df = mysql_top_p95
    kpi_anoms_used = kpi_anoms_p95
    thresholds_used = thresholds_p95
    relaxed_to = 'P95'

# Limit component summary to selected components (already restricted) and up to 14 rows
comp_summary_head = comp_summary.head(14)

# Limit mysql_top_df to up to 2 rows
mysql_top_df_head = mysql_top_df.head(2)

# Final outputs: compact tables and which threshold was used
comp_summary_head, mysql_top_df_head, relaxed_to

```
Out[4]:
```
summary_lines = []
try:
    comps = comp_summary_head
    mysql_top = mysql_top_df_head
    thr = relaxed_to
except NameError:
    summary = "No results available in the kernel to summarize."
else:
    summary_lines.append(f"Metric anomaly summary for selected components within 2021-03-07 21:30–22:00 (Asia/Shanghai).")
    summary_lines.append(f"Global threshold used: {thr} (computed from full series before window filtering).")
    summary_lines.append("")
    summary_lines.append("Component-level highlights (earliest anomaly time, total anomaly points, #KPIs, max severity%):")
    for _, r in comps.iterrows():
        summary_lines.append(
            f"- {r['cmdb_id']}: earliest={r['earliest_anomaly_timestamp']}, "
            f"total_points={int(r['total_anomaly_points_in_window'])}, KPIs={int(r['number_of_KPIs_with_anomalies'])}, "
            f"max_severity={r['max_severity_pct_across_KPIs']:.1f}%"
        )
    summary_lines.append("")
    summary_lines.append("Top KPI per Mysql host (largest severity_pct among KPIs with anomalies):")
    if mysql_top.empty:
        summary_lines.append("- No KPI-level anomalies found for Mysql01/Mysql02 with the chosen threshold.")
    else:
        for _, r in mysql_top.iterrows():
            summary_lines.append(
                f"- {r['cmdb_id']} | {r['kpi_name']}: earliest={r['earliest_anomaly_timestamp_for_this_kpi']}, "
                f"max_in_window={r['max_value_in_window']}, p95={r['p95_value']}, severity={r['severity_pct']:.1f}%"
            )
    summary_lines.append("")
    summary_lines.append("High-level takeaway:")
    summary_lines.append(
        "Multiple core components show numerous KPI anomalies during the incident window. "
        "Redis02 and Redis01 show very large anomaly counts and the highest single-KPI severity (Redis02 ~2113%). "
        "Mysql01 and Mysql02 also have very large anomaly point totals (Mysql01 ~2533, Mysql02 ~2114) — "
        "Mysql01's top anomalous KPI is InnoDB Row Lock Time (severity ~499%), indicating heavy DB contention. "
        "Other infra (apache, Tomcat, IG/MG) also show significant anomaly activity. "
        "These results use P95 thresholds computed on the full series."
    )
    summary = "\n".join(summary_lines)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id earliest_anomaly_timestamp  total_anomaly_points_in_window  number_of_KPIs_with_anomalies  max_severity_pct_across_KPIs
0    Redis02   2021-03-07 21:30:00+0800                            1364                            107                   2113.906188
1    Mysql01   2021-03-07 21:30:00+0800                            2533                            221                    499.321220
2   apache02   2021-03-07 21:30:00+0800                             615                             81                    263.636364
3   Tomcat02   2021-03-07 21:30:00+0800                             967                            107                    237.580336
4    Mysql02   2021-03-07 21:30:00+0800                            2114                            215                    233.333333
5       IG01   2021-03-07 21:30:00+0800                             624                             89                    233.333333
6   Tomcat03   2021-03-07 21:30:00+0800                             962                            108                    216.500776
7   Tomcat01   2021-03-07 21:30:00+0800                             866                            105                    150.000000
8   apache01   2021-03-07 21:30:00+0800                             150                             37                    149.203518
9    Redis01   2021-03-07 21:30:00+0800                            1335                            109                    139.903846
10      IG02   2021-03-07 21:30:00+0800                             570                             83                    100.055529
11  Tomcat04   2021-03-07 21:30:00+0800                            1025                            110                    100.000000
12      MG02   2021-03-07 21:30:00+0800                             667                             91                     93.594771
13      MG01   2021-03-07 21:30:00+0800                             676                             89                     81.818182,    cmdb_id                                           kpi_name earliest_anomaly_timestamp_for_this_kpi  max_value_in_window  p95_value  severity_pct
0  Mysql01              Mysql-MySQL_3306_Innodb Row Lock Time                2021-03-07 21:46:00+0800              16.3167    2.72253    499.321220
1  Mysql02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKPer...                2021-03-07 21:55:00+0800               0.8000    0.24000    233.333333, 'P95')```
```